# 기상 데이터 수집

이 파일은 기상청 API Hub를 활용하여 빅콘테스트 분석에 필요한 기상 데이터를 수집하기 위한 파일입니다.

분석 대상 지역인 서울 강남구와 강원 춘천시의 기상 데이터를 수집하며,
Raw 데이터베이스 적재 전에 API 응답 구조와 수집 결과를 검증할 수 있도록 구성하였습니다.

## 주요 내용
- 기상청 API 인증정보 설정
- 분석 대상 지역 및 기상요소 정의
- 기상청 API 호출 함수 정의
- 서울 강남구 및 강원 춘천시 일별 기상 데이터 수집
- 수집된 데이터 구조 및 행 수 확인
- Raw 데이터 적재를 위한 원본 데이터 구성

## 결과
- 2025년 7월부터 12월까지의 일별 기상 데이터 수집
- 서울 강남구 및 강원 춘천시 기상 데이터 확보
- 최고기온, 최저기온, 일강수량, 최대풍속, 최대순간풍속 데이터 확보
- Raw 데이터베이스 적재를 위한 기상 원본 데이터 생성

---

## 1. Imports & Settings

기상 데이터 수집에 필요한 라이브러리와 환경변수를 불러오고,
API 호출에 사용되는 전역 설정값 및 공통 함수를 정의합니다.

### 주요 작업
- 기상 데이터 수집에 필요한 라이브러리 로드
- `.env` 파일에서 기상청 API 인증키 로드
- API URL 및 수집 기간 설정
- 대상 관측소 및 기상요소 정의
- 공통 기상 데이터 수집 함수 정의

In [34]:
# 라이브러리
import os
import time

import pandas as pd
import requests

from dotenv import load_dotenv

from pathlib import Path

In [2]:
# 환경변수 로드
load_dotenv()

# 기상청 API 인증키 로드
AUTH_KEY = os.getenv("KMA_API_KEY")

# 인증키 로드 여부 확인
if AUTH_KEY is None:
    raise ValueError("KMA_API_KEY를 불러오지 못했습니다.")

print("기상청 API 인증키 로드 완료")

기상청 API 인증키 로드 완료


In [3]:
# API 기본 설정
BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/sfc_aws_day.php"
AWS_HOUR_URL = "https://apihub.kma.go.kr/api/typ01/url/awsh.php"
AWS_MIN_URL = "https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph-aws2_min"

# 수집 기간
START_DATE = "20250701"
END_DATE = "20251231"

# 대상 관측소
STATIONS = {
    400: "서울 강남구",
    101: "강원 춘천시"
}

# 수집 기상요소
OBSERVATIONS = {
    "ta_max": "일 최고기온",
    "ta_min": "일 최저기온",
    "rn_day": "일 강수량",
    "ws_max": "일 최대풍속",
    "ws_ins_max": "일 최대순간풍속"
}

print(f"수집 기간: {START_DATE} ~ {END_DATE}")
print(f"대상 지역 수: {len(STATIONS)}")
print(f"기상요소 수: {len(OBSERVATIONS)}")

print("\n[대상 관측소]")
for stn, region in STATIONS.items():
    print(f"- {region}: {stn}")

print("\n[기상요소]")
for code, name in OBSERVATIONS.items():
    print(f"- {code}: {name}")

수집 기간: 20250701 ~ 20251231
대상 지역 수: 2
기상요소 수: 5

[대상 관측소]
- 서울 강남구: 400
- 강원 춘천시: 101

[기상요소]
- ta_max: 일 최고기온
- ta_min: 일 최저기온
- rn_day: 일 강수량
- ws_max: 일 최대풍속
- ws_ins_max: 일 최대순간풍속


In [30]:
def get_weather_data(stn, obs_code):
    params = {
        "tm1": START_DATE,
        "tm2": END_DATE,
        "obs": obs_code,
        "stn": str(stn),
        "disp": "0",
        "help": "0",
        "authKey": AUTH_KEY
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    rows = []

    for line in response.text.splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        values = line.split(maxsplit=6)

        if len(values) < 7:
            continue

        rows.append({
            "TM": values[0],
            "STN": values[1],
            "LON": values[2],
            "LAT": values[3],
            "HT": values[4],
            "VAL": values[5],
            "STN_NAME": values[6],
            "OBS_CODE": obs_code,
            "OBS_NAME": OBSERVATIONS[obs_code]
        })

    return pd.DataFrame(rows)


def check_aws_minute(tm, stn=400):
    params = {
        "tm2": tm,
        "stn": str(stn),
        "disp": "0",
        "help": "0",
        "authKey": AUTH_KEY
    }

    try:
        response = requests.get(
            AWS_MIN_URL,
            params=params,
            timeout=(10, 30)
        )

        # 정상 응답
        if response.status_code == 200:
            data_lines = [
                line.strip()
                for line in response.text.splitlines()
                if line.strip() and not line.startswith("#")
            ]

            return {
                "TM": tm,
                "STATUS": 200,
                "RESULT": data_lines[0] if data_lines else "데이터 행 없음"
            }

        # 200 이외 응답
        return {
            "TM": tm,
            "STATUS": response.status_code,
            "RESULT": "정상 응답 아님"
        }

    except requests.exceptions.RequestException as e:
        return {
            "TM": tm,
            "STATUS": None,
            "RESULT": type(e).__name__
        }


def get_aws_minute_range(stn, tm1, tm2, max_retries=5):
    params = {
        "tm1": tm1,
        "tm2": tm2,
        "stn": str(stn),
        "disp": "0",
        "help": "0",
        "authKey": AUTH_KEY
    }

    # 요청 구간의 예상 행 수
    start_dt = pd.to_datetime(tm1, format="%Y%m%d%H%M")
    end_dt = pd.to_datetime(tm2, format="%Y%m%d%H%M")

    expected_count = int(
        (end_dt - start_dt).total_seconds() // 60
    ) + 1

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                AWS_MIN_URL,
                params=params,
                timeout=(10, 120)
            )

            response.raise_for_status()

            rows = []

            for line in response.text.splitlines():
                line = line.strip()

                if not line or line.startswith("#"):
                    continue

                values = line.split()

                if len(values) < 18:
                    continue

                rows.append({
                    "TM": values[0],
                    "STN": values[1],
                    "WD1": values[2],
                    "WS1": values[3],
                    "WDS": values[4],
                    "WSS": values[5],
                    "WD10": values[6],
                    "WS10": values[7],
                    "TA": values[8],
                    "RE": values[9],
                    "RN_15M": values[10],
                    "RN_60M": values[11],
                    "RN_12H": values[12],
                    "RN_DAY": values[13],
                    "HM": values[14],
                    "PA": values[15],
                    "PS": values[16],
                    "TD": values[17]
                })

            # 데이터가 일부 또는 전체 누락된 경우 재시도
            if len(rows) != expected_count:
                raise RuntimeError(
                    f"응답 행 수 불일치: "
                    f"{len(rows)}/{expected_count}"
                )

            return pd.DataFrame(rows)

        except (
            requests.exceptions.RequestException,
            RuntimeError
        ):
            if attempt == max_retries:
                raise

            time.sleep(attempt * 3)
            

def get_aws_minute_day(stn, date):
    # 카드 데이터 시간대와 동일하게 6시간 단위로 분할하여 수집
    time_ranges = [
        (f"{date}0000", f"{date}0559"),
        (f"{date}0600", f"{date}1159"),
        (f"{date}1200", f"{date}1759"),
        (f"{date}1800", f"{date}2359")
    ]

    day_data = []

    for tm1, tm2 in time_ranges:
        df = get_aws_minute_range(
            stn=stn,
            tm1=tm1,
            tm2=tm2
        )

        day_data.append(df)

        # 연속 API 요청 방지
        time.sleep(0.3)

    return pd.concat(
        day_data,
        ignore_index=True
    )

TIME_BLOCK_RANGES = {
    "00_05": ("0000", "0559"),
    "06_11": ("0600", "1159"),
    "12_17": ("1200", "1759"),
    "18_23": ("1800", "2359")
}

In [39]:
# 저장 경로
# 프로젝트 루트 탐색
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "bigcontest":
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("bigcontest 프로젝트 루트를 찾을 수 없습니다.")

    PROJECT_ROOT = PROJECT_ROOT.parent

# 기상 원본 데이터 저장 경로
WEATHER_RAW_DIR = (
    PROJECT_ROOT
    / "00_data"
    / "weather"
)

WEATHER_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("기상 데이터 저장 경로:", WEATHER_RAW_DIR)

기상 데이터 저장 경로: /Users/lee-jongyoon/Documents/bigcontest/00_data/weather


## 2. API 수집 테스트

전체 기상 데이터를 수집하기 전에 단일 관측소와 단일 기상요소를 대상으로
수집 함수가 정상적으로 동작하는지 확인합니다.

서울 강남구 관측소의 일 최고기온과 시간별 최고기온 데이터를 조회하여
수집 기간, 컬럼 구조, 행 수가 예상한 형태로 반환되는지 검증합니다.

In [5]:
# 서울 강남구 일 최고기온 테스트 수집
test_weather = get_weather_data(
    stn=400,
    obs_code="ta_max"
)

test_weather.head()

,TM,STN,LON,LAT,HT,VAL,STN_NAME,OBS_CODE,OBS_NAME
0,20250701,400,127.08162000,37.49820000,12.66,33.3,강남,ta_max,일 최고기온
1,20250702,400,127.08162000,37.49820000,12.66,32.5,강남,ta_max,일 최고기온
2,20250703,400,127.08162000,37.49820000,12.66,33.1,강남,ta_max,일 최고기온
3,20250704,400,127.08162000,37.49820000,12.66,31.5,강남,ta_max,일 최고기온
4,20250705,400,127.08162000,37.49820000,12.66,33.1,강남,ta_max,일 최고기온


In [6]:
print("수집 행 수:", len(test_weather))
print("수집 시작일:", test_weather["TM"].min())
print("수집 종료일:", test_weather["TM"].max())

print("\n컬럼 목록:")
print(test_weather.columns.tolist())

수집 행 수: 184
수집 시작일: 20250701
수집 종료일: 20251231

컬럼 목록:
['TM', 'STN', 'LON', 'LAT', 'HT', 'VAL', 'STN_NAME', 'OBS_CODE', 'OBS_NAME']


In [7]:
params = {
    "tm": "202507011200",
    "stn": "400",
    "help": "1",
    "authKey": AUTH_KEY
}

response = requests.get(
    AWS_HOUR_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

print(response.text)

#START7777
#--------------------------------------------------------------------------------------------------
#  TA     : 정시기온 (C) 
#  WD     : 정시 10분 평균 풍향 (degee) 
#  WS     : 정시 10분 평균 풍속 (m/s) 
#  RN_DAY : 해당시간까지의 일강수량 (mm) 
#  RN_HR1 : 1시간 강수량 (mm) 
#  HM     : 정시습도 (%) 
#  PA     : 정시 현지기압 (hPa) 
#  PS     : 정시 해면기압 (hPa) 
#--------------------------------------------------------------------------------------------------
# YYMMDDHHMI   STN     TA     WD     WS RN_DAY RN_HR1     HM     PA     PS
#        KST    ID      C    deg    m/s     mm     mm      %    hPa    hPa
202507011200   400   31.4  195.5    2.4    0.0    0.0   65.9  -99.0  -99.0
#7777END



## 3. 일별 기상 데이터 전체 수집

설정한 관측소와 기상요소 조합을 순회하며
서울 강남구와 강원 춘천시의 일별 기상 데이터를 전체 수집합니다.

각 API 요청 결과를 DataFrame으로 저장한 뒤 하나의 데이터셋으로 결합하며,
API 서버에 과도한 요청이 발생하지 않도록 요청 사이에 짧은 대기 시간을 적용합니다.

In [9]:
# 전체 기상 데이터 수집
weather_list = []

for stn, region in STATIONS.items():
    for obs_code, obs_name in OBSERVATIONS.items():

        print(f"수집 중: {region} / {obs_name}({obs_code})")

        df = get_weather_data(
            stn=stn,
            obs_code=obs_code
        )

        weather_list.append(df)

        # API 연속 호출 방지를 위한 짧은 대기
        time.sleep(0.2)

수집 중: 서울 강남구 / 일 최고기온(ta_max)
수집 중: 서울 강남구 / 일 최저기온(ta_min)
수집 중: 서울 강남구 / 일 강수량(rn_day)
수집 중: 서울 강남구 / 일 최대풍속(ws_max)
수집 중: 서울 강남구 / 일 최대순간풍속(ws_ins_max)
수집 중: 강원 춘천시 / 일 최고기온(ta_max)
수집 중: 강원 춘천시 / 일 최저기온(ta_min)
수집 중: 강원 춘천시 / 일 강수량(rn_day)
수집 중: 강원 춘천시 / 일 최대풍속(ws_max)
수집 중: 강원 춘천시 / 일 최대순간풍속(ws_ins_max)


In [10]:
# 수집 결과 결합
weather_raw = pd.concat(
    weather_list,
    ignore_index=True
)

weather_raw.head()

,TM,STN,LON,LAT,HT,VAL,STN_NAME,OBS_CODE,OBS_NAME
0,20250701,400,127.08162000,37.49820000,12.66,33.3,강남,ta_max,일 최고기온
1,20250702,400,127.08162000,37.49820000,12.66,32.5,강남,ta_max,일 최고기온
2,20250703,400,127.08162000,37.49820000,12.66,33.1,강남,ta_max,일 최고기온
3,20250704,400,127.08162000,37.49820000,12.66,31.5,강남,ta_max,일 최고기온
4,20250705,400,127.08162000,37.49820000,12.66,33.1,강남,ta_max,일 최고기온


In [11]:
print("전체 수집 행, 열 수:", weather_raw.shape)

전체 수집 행, 열 수: (1839, 9)


## 4. 일별 기상 데이터 수집 결과 검증

수집된 일별 기상 데이터가 예상한 기간과 관측소, 기상요소별로 정상적으로 구성되었는지 검증합니다.

전체 예상 행 수와 실제 수집 행 수를 비교하고,
관측소 및 기상요소별 수집 건수를 확인하여 누락된 데이터가 있는지 점검합니다.

In [12]:
# 관측소 및 기상요소별 수집 행 수 확인
weather_count = (
    weather_raw
    .groupby(
        ["STN", "STN_NAME", "OBS_CODE", "OBS_NAME"]
    )
    .size()
    .reset_index(name="ROW_COUNT")
)

weather_count

,STN,STN_NAME,OBS_CODE,OBS_NAME,ROW_COUNT
0,101,춘천,rn_day,일 강수량,184
1,101,춘천,ta_max,일 최고기온,184
2,101,춘천,ta_min,일 최저기온,184
3,101,춘천,ws_ins_max,일 최대순간풍속,184
4,101,춘천,ws_max,일 최대풍속,184
5,400,강남,rn_day,일 강수량,183
6,400,강남,ta_max,일 최고기온,184
7,400,강남,ta_min,일 최저기온,184
8,400,강남,ws_ins_max,일 최대순간풍속,184
9,400,강남,ws_max,일 최대풍속,184


In [13]:
# 예상 일수
expected_days = len(
    pd.date_range(
        start=pd.to_datetime(START_DATE),
        end=pd.to_datetime(END_DATE)
    )
)

# 수집 건수가 예상 일수와 다른 항목 확인
weather_count[
    weather_count["ROW_COUNT"] != expected_days
]

,STN,STN_NAME,OBS_CODE,OBS_NAME,ROW_COUNT
5,400,강남,rn_day,일 강수량,183


### 4.1 누락 일별 기상 데이터 확인

관측소 및 기상요소별 수집 건수를 확인한 결과,
서울 강남구의 일 강수량(`rn_day`) 데이터가 예상 일수보다 1건 적게 수집되었습니다.

누락된 값을 임의로 보정하기 전에
전체 분석 기간과 실제 수집 날짜를 비교하여 누락된 날짜를 확인합니다.

In [14]:
# 전체 분석 기간 날짜 생성
expected_dates = set(
    pd.date_range(
        start=pd.to_datetime(START_DATE),
        end=pd.to_datetime(END_DATE)
    ).strftime("%Y%m%d")
)

# 강남 일 강수량 데이터의 실제 날짜
actual_dates = set(
    weather_raw.loc[
        (weather_raw["STN"] == "400")
        & (weather_raw["OBS_CODE"] == "rn_day"),
        "TM"
    ]
)

# 누락 날짜 확인
missing_dates = sorted(expected_dates - actual_dates)

print("누락 날짜:", missing_dates)
print("누락 건수:", len(missing_dates))

누락 날짜: ['20250911']
누락 건수: 1


### 4.2 누락 데이터 단건 재조회

누락된 2025년 9월 11일 서울 강남구의 일 강수량 데이터를
기상청 API에서 단건으로 다시 조회합니다.

전체 기간 수집 과정의 일시적인 오류인지,
기상청 원본 데이터 자체에 관측값이 존재하지 않는지를 확인합니다.

In [15]:
# 누락 데이터 단건 재조회
params = {
    "tm1": "20250911",
    "tm2": "20250911",
    "obs": "rn_day",
    "stn": "400",
    "disp": "0",
    "help": "1",
    "authKey": AUTH_KEY
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

print(response.text)

#START7777
#--------------------------------------------------------------------------------------------------
#  기상청 지상관측 시간자료
#--------------------------------------------------------------------------------------------------
#  1. TM     : 관측시각 (KST)
#  2. STN    : 국내 지점번호
#  3. LON    : 경도 (deg)
#  4. LAT    : 위도 (deg)
#  5. HT     : 노장 해발고도 (m)
#  6. VAL    : 조회된 값
#--------------------------------------------------------------------------------------------------
# YYMMDD   STN         LON          LAT       HT      VAL
#    KST    ID        (deg)        (deg)     (m)         
#7777END



#### 확인 결과

서울 강남구 관측소(400)의 2025년 9월 11일 일 강수량(`rn_day`)을
기상청 API에서 단건으로 재조회한 결과, 해당 날짜의 관측 데이터가 반환되지 않았습니다.

따라서 전체 수집 데이터의 1건 누락은 수집 로직의 오류가 아닌
기상청 API 원본 데이터의 미제공 값으로 판단합니다.

Raw 단계에서는 원본 데이터를 임의로 생성하거나 보정하지 않고,
수집된 1,839건을 그대로 유지합니다.

해당 누락값은 Staging 단계에서 전체 날짜 기준으로 정렬한 후
결측값(`NULL`)으로 관리합니다.

### 4.3 누락일 AWS 매분자료 확인

일 강수량이 누락된 2025년 9월 11일 서울 강남구 관측소의
AWS 매분자료를 조회하여 강수 관련 원시 관측값의 존재 여부를 확인합니다.

일통계 자료에서 누락된 날짜의 매분자료를 직접 조회함으로써,
강수 관측값 자체의 부재인지 또는 일통계 생성 과정의 문제인지 점검합니다.

In [16]:
# params = {
#     "tm2": "202509111200",
#     "stn": "400",
#     "disp": "0",
#     "help": "0",
#     "authKey": AUTH_KEY
# }

# response = requests.get(
#     AWS_MIN_URL,
#     params=params,
#     timeout=(10, 120)
# )

# response.raise_for_status()

# print(response.text)

#### 확인 결과

2025년 9월 11일 12시 서울 강남구 관측소(400)의 AWS 매분자료를 조회한 결과,
기상청 API에서 `504 Gateway Timeout`이 발생하였습니다.

동일한 API를 이용한 다른 날짜의 조회는 정상적으로 수행되었으므로,
해당 날짜 또는 해당 시점의 자료 조회 과정에 문제가 있을 가능성이 있습니다.

정확한 원인을 확인하기 위해 누락일 전후 시점의 AWS 매분자료를 추가로 비교합니다.

### 4.4 누락일 전후 AWS 자료 비교

2025년 9월 11일 서울 강남구 AWS 매분자료 조회에서 발생한 오류가
특정 시점의 문제인지 해당 날짜 전체의 문제인지 확인합니다.

누락일의 전날과 다음날 동일 시간의 자료를 함께 조회하고,
누락일 내부에서도 자정·정오·마지막 시점의 자료를 비교합니다.

### 비교 시점
- 2025-09-10 12:00: 누락일 전날
- 2025-09-11 00:00: 누락일 시작 시점
- 2025-09-11 12:00: 누락일 정오
- 2025-09-11 23:59: 누락일 마지막 시점
- 2025-09-12 12:00: 누락일 다음날

In [17]:
# 누락일 전후 비교 시점
test_times = [
    "202509101200",
    "202509110000",
    "202509111200",
    "202509112359",
    "202509121200"
]

# AWS 매분자료 비교
aws_check_results = []

for tm in test_times:
    result = check_aws_minute(tm)

    aws_check_results.append(result)

    # API 연속 호출 방지
    time.sleep(0.5)

aws_check = pd.DataFrame(aws_check_results)

aws_check

,TM,STATUS,RESULT
0,202509101200,200,202509101200 400 -99.9 -99.9 -99.9 -99.9...
1,202509110000,200,202509110000 400 -99.9 -99.9 -99.9 -99.9...
2,202509111200,200,202509111200 400 -99.9 -99.9 -99.9 -99.9...
3,202509112359,200,202509112359 400 164.1 0.3 169.5 0.9...
4,202509121200,200,202509121200 400 131.4 1.3 80.7 2.3...


In [18]:
# 정상 응답 데이터에서 강수 관련 항목 추출
rain_check = []

for _, row in aws_check.iterrows():
    if row["STATUS"] != 200:
        continue

    values = row["RESULT"].split()

    rain_check.append({
        "TM": values[0],
        "STN": values[1],
        "RE": values[9],
        "RN_15M": values[10],
        "RN_60M": values[11],
        "RN_12H": values[12],
        "RN_DAY": values[13]
    })

rain_check = pd.DataFrame(rain_check)

rain_check

,TM,STN,RE,RN_15M,RN_60M,RN_12H,RN_DAY
0,202509101200,400,-99.9,-99.9,-99.9,-99.9,-99.9
1,202509110000,400,-99.9,-99.9,-99.9,-99.9,-99.9
2,202509111200,400,-99.9,-99.9,-99.9,-99.9,-99.9
3,202509112359,400,0.0,-99.9,-99.9,-99.9,-99.2
4,202509121200,400,0.0,0.0,0.0,0.0,0.0


#### 확인 결과

2025년 9월 11일 서울 강남구 관측소(400)의 AWS 매분자료를
누락일 전후 시점과 비교한 결과, 9월 10일과 9월 11일 정오의
강수 관련 관측값이 모두 `-99.9`로 반환되었습니다.

기상청 자료에서 `-99.9`는 결측값을 의미합니다.

그러나 9월 10일의 일 강수량 자료는 일통계 API에서 정상적으로 제공되고 있으므로,
특정 시점의 매분 강수자료 결측만으로 9월 11일 일 강수량 누락의 원인을
직접 판단하기는 어렵습니다.

따라서 현재 단계에서는 2025년 9월 11일 서울 강남구의 일 강수량이
기상청 일통계 API에서 제공되지 않는 원본 결측값임을 확인한 것으로 정리하고,
Raw 단계에서는 해당 값을 임의로 보정하지 않습니다.

### 4.5 중복 데이터 확인

수집된 기상 데이터에서 동일한 날짜, 관측소, 기상요소 조합이
중복으로 존재하는지 확인합니다.

일별 기상 데이터는 날짜(`TM`), 관측소(`STN`), 기상요소(`OBS_CODE`)의
조합이 하나의 관측값을 식별하므로 해당 컬럼을 기준으로 중복 여부를 검증합니다.

In [19]:
# 날짜 + 관측소 + 기상요소 기준 중복 확인
duplicate_mask = weather_raw.duplicated(
    subset=["TM", "STN", "OBS_CODE"],
    keep=False
)

duplicate_rows = weather_raw[duplicate_mask]

print("중복 행 수:", len(duplicate_rows))

duplicate_rows

중복 행 수: 0


,TM,STN,LON,LAT,HT,VAL,STN_NAME,OBS_CODE,OBS_NAME


In [20]:
# 전체 컬럼 기준 완전 중복 확인
print(
    "전체 컬럼 기준 중복 행 수:",
    weather_raw.duplicated().sum()
)

전체 컬럼 기준 중복 행 수: 0


#### 확인 결과

날짜(`TM`), 관측소(`STN`), 기상요소(`OBS_CODE`) 조합을 기준으로
중복 데이터를 확인한 결과 중복 행은 존재하지 않았습니다.

전체 컬럼이 동일한 완전 중복 데이터 역시 확인되지 않았습니다.

### 4.6 컬럼 구조 및 데이터 타입 확인

수집된 기상 데이터의 컬럼 구성과 데이터 타입을 확인합니다.

Raw 단계에서는 원본 값을 임의로 변환하지 않고 현재 수집 상태를 유지하며,
이후 Staging 테이블 설계를 위해 날짜, 지점번호, 좌표, 관측값 등의
현재 데이터 타입을 확인합니다.

In [21]:
# 컬럼 구조 및 데이터 타입 확인
weather_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1839 entries, 0 to 1838
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   TM        1839 non-null   object
 1   STN       1839 non-null   object
 2   LON       1839 non-null   object
 3   LAT       1839 non-null   object
 4   HT        1839 non-null   object
 5   VAL       1839 non-null   object
 6   STN_NAME  1839 non-null   object
 7   OBS_CODE  1839 non-null   object
 8   OBS_NAME  1839 non-null   object
dtypes: object(9)
memory usage: 129.4+ KB


In [22]:
# 컬럼별 결측값 확인
weather_raw.isna().sum()

TM          0
STN         0
LON         0
LAT         0
HT          0
VAL         0
STN_NAME    0
OBS_CODE    0
OBS_NAME    0
dtype: int64

#### 확인 결과

수집된 기상 데이터의 각 컬럼은 API 응답을 원본 형태로 보존하기 위해
문자열(`object`) 중심으로 구성되어 있습니다.

수집된 1,839개 행 내부에서는 컬럼 단위 결측값이 확인되지 않았습니다.

다만 앞선 검증에서 서울 강남구의 2025년 9월 11일 일 강수량(`rn_day`)은
API에서 행 자체가 제공되지 않는 원본 결측으로 확인되었으므로,
일반적인 `NaN` 검사와 별도로 관리합니다.

데이터 타입 변환은 Raw 단계가 아닌 Staging 단계에서 수행합니다.

## 5. 시간 단위 기상 데이터 전체 수집

카드 데이터의 시간대별 소비 정보와 결합하기 위해
서울 강남구와 강원 춘천시의 AWS 매분자료를 수집합니다.

AWS 매분자료는 한 개 관측소에 대해 하루 이내의 기간 조회가 가능하므로,
관측소와 날짜를 기준으로 API를 반복 호출하여 전체 기간의 자료를 수집합니다.

수집된 매분자료는 원본 형태로 저장하며,
카드 데이터의 6시간 단위 시간대와의 결합을 위한 집계는 이후 전처리 단계에서 수행합니다.

In [23]:
minute_test = get_aws_minute_day(
    stn=400,
    date="20250701"
)

print("전체 행 수:", len(minute_test))
print(
    "시간 범위:",
    minute_test["TM"].min(),
    "~",
    minute_test["TM"].max()
)

minute_test.head()

재시도 중: 400 / 202507011800~202507012359 (1/5)
재시도 중: 400 / 202507011800~202507012359 (2/5)
전체 행 수: 1440
시간 범위: 202507010000 ~ 202507012359


,TM,STN,WD1,WS1,WDS,WSS,WD10,WS10,TA,RE,RN_15M,RN_60M,RN_12H,RN_DAY,HM,PA,PS,TD
0,202507010000,400,183.5,1.7,199.7,2.0,188.9,1.2,27.5,0.0,0.0,0.0,0.0,0.0,74.6,-99.7,-99.9,22.6
1,202507010001,400,195.0,1.7,185.6,2.4,189.7,1.2,27.5,0.0,0.0,0.0,0.0,0.0,74.5,-99.7,-99.9,22.6
2,202507010002,400,199.7,1.4,191.3,1.7,190.4,1.1,27.6,0.0,0.0,0.0,0.0,0.0,74.5,-99.7,-99.9,22.7
3,202507010003,400,162.4,0.8,182.8,1.1,188.9,1.1,27.6,0.0,0.0,0.0,0.0,0.0,74.4,-99.7,-99.9,22.6
4,202507010004,400,179.2,1.2,208.1,2.1,188.6,1.2,27.6,0.0,0.0,0.0,0.0,0.0,74.5,-99.7,-99.9,22.7


In [24]:
# 전체 수집 대상 날짜
weather_dates = pd.date_range(
    start=pd.to_datetime(START_DATE),
    end=pd.to_datetime(END_DATE),
    freq="D"
)

minute_list = []
failed_collects = []

for stn, region in STATIONS.items():
    for i, date in enumerate(weather_dates, start=1):
        date_str = date.strftime("%Y%m%d")

        print(
            f"[{region}] "
            f"{date_str} "
            f"({i}/{len(weather_dates)})"
        )

        try:
            df = get_aws_minute_day(
                stn=stn,
                date=date_str
            )

            minute_list.append(df)

        except requests.exceptions.RequestException as e:
            failed_collects.append({
                "STN": str(stn),
                "REGION": region,
                "DATE": date_str,
                "ERROR": type(e).__name__
            })

        time.sleep(0.5)

weather_minute_raw = pd.concat(
    minute_list,
    ignore_index=True
)

[서울 강남구] 20250701 (1/184)
재시도 중: 400 / 202507010000~202507010559 (1/5)
재시도 중: 400 / 202507010600~202507011159 (1/5)
[서울 강남구] 20250702 (2/184)
[서울 강남구] 20250703 (3/184)
재시도 중: 400 / 202507030600~202507031159 (1/5)
재시도 중: 400 / 202507031200~202507031759 (1/5)
재시도 중: 400 / 202507031800~202507032359 (1/5)
[서울 강남구] 20250704 (4/184)
재시도 중: 400 / 202507041800~202507042359 (1/5)
재시도 중: 400 / 202507041800~202507042359 (2/5)
[서울 강남구] 20250705 (5/184)
[서울 강남구] 20250706 (6/184)
재시도 중: 400 / 202507061800~202507062359 (1/5)
재시도 중: 400 / 202507061800~202507062359 (2/5)
[서울 강남구] 20250707 (7/184)
재시도 중: 400 / 202507070600~202507071159 (1/5)
[서울 강남구] 20250708 (8/184)
[서울 강남구] 20250709 (9/184)
[서울 강남구] 20250710 (10/184)
[서울 강남구] 20250711 (11/184)
[서울 강남구] 20250712 (12/184)
재시도 중: 400 / 202507121800~202507122359 (1/5)
재시도 중: 400 / 202507121800~202507122359 (2/5)
[서울 강남구] 20250713 (13/184)
[서울 강남구] 20250714 (14/184)
[서울 강남구] 20250715 (15/184)
[서울 강남구] 20250716 (16/184)
[서울 강남구] 20250717 (17/184)
[서울 강남구] 2

In [25]:
print("전체 수집 행 수:", len(weather_minute_raw))
print("수집 실패 건수:", len(failed_collects))

print(
    "시간 범위:",
    weather_minute_raw["TM"].min(),
    "~",
    weather_minute_raw["TM"].max()
)

print(
    "관측소:",
    weather_minute_raw["STN"].unique()
)

전체 수집 행 수: 483480
수집 실패 건수: 1
시간 범위: 202507010000 ~ 202512312359
관측소: ['400' '101']


## 6. 시간 단위 기상 데이터 수집 결과 검증

수집된 AWS 매분자료가 대상 기간과 관측소에 대해 정상적으로 확보되었는지 확인합니다.

전체 예상 행 수와 실제 수집 행 수를 비교하고,
관측소·날짜별 데이터 개수 및 API 요청 실패 내역을 확인하여
누락된 시간 구간이 존재하는지 검증합니다.

In [26]:
# 관측소별 수집 행 수 확인
station_count = (
    weather_minute_raw
    .groupby("STN")
    .size()
    .reset_index(name="ROW_COUNT")
)

station_count

,STN,ROW_COUNT
0,101,238680
1,400,244800


In [27]:
# API 요청 실패 내역 확인
failed_df = pd.DataFrame(failed_collects)

failed_df

,STN,REGION,DATE,ERROR
0,400,서울 강남구,20250806,HTTPError


In [28]:
# 날짜 컬럼 생성
minute_check = weather_minute_raw.copy()
minute_check["DATE"] = minute_check["TM"].str[:8]

# 관측소 + 날짜별 수집 행 수
daily_count = (
    minute_check
    .groupby(["STN", "DATE"])
    .size()
    .reset_index(name="ROW_COUNT")
)

# 1,440행이 아닌 날짜만 확인
incomplete_days = daily_count[
    daily_count["ROW_COUNT"] != 1440
]

print("불완전한 관측소-날짜 수:", len(incomplete_days))

incomplete_days

불완전한 관측소-날짜 수: 106


,STN,DATE,ROW_COUNT
6,101,20250707,1080
8,101,20250709,360
12,101,20250713,720
14,101,20250715,1080
15,101,20250716,1080
...,...,...,...
344,400,20251209,1080
346,400,20251211,1080
347,400,20251212,1080
356,400,20251221,1080


### 6.2 시간 구간별 수집 현황 확인

AWS 매분자료는 하루를 6시간 단위로 분할하여 수집하므로,
관측소·날짜·시간구간별 데이터 개수를 확인합니다.

각 6시간 구간에는 정상적으로 360개의 매분자료가 존재해야 하며,
기대되는 전체 조합을 생성하여 부분 누락뿐만 아니라
하루 전체가 수집되지 않은 경우도 함께 확인합니다.

In [29]:
# 검증용 데이터 생성
minute_check = weather_minute_raw[["TM", "STN"]].copy()

minute_check["DATE"] = minute_check["TM"].str[:8]
minute_check["HOUR"] = minute_check["TM"].str[8:10].astype(int)

# 카드 데이터와 동일한 6시간 단위 구간 생성
minute_check["TIME_BLOCK"] = (
    minute_check["HOUR"] // 6
).map({
    0: "00_05",
    1: "06_11",
    2: "12_17",
    3: "18_23"
})

# 실제 구간별 행 수
block_count = (
    minute_check
    .groupby(["STN", "DATE", "TIME_BLOCK"])
    .size()
)

# 기대되는 전체 관측소 × 날짜 × 시간구간 조합
expected_dates = pd.date_range(
    start=pd.to_datetime(START_DATE),
    end=pd.to_datetime(END_DATE),
    freq="D"
).strftime("%Y%m%d")

expected_index = pd.MultiIndex.from_product(
    [
        [str(stn) for stn in STATIONS.keys()],
        expected_dates,
        ["00_05", "06_11", "12_17", "18_23"]
    ],
    names=["STN", "DATE", "TIME_BLOCK"]
)

# 실제 데이터가 없는 구간은 0으로 처리
block_count = (
    block_count
    .reindex(expected_index, fill_value=0)
    .reset_index(name="ROW_COUNT")
)

# 360행이 아닌 구간 확인
incomplete_blocks = block_count[
    block_count["ROW_COUNT"] != 360
].reset_index(drop=True)

print("전체 기대 구간 수:", len(expected_index))
print("불완전한 시간 구간 수:", len(incomplete_blocks))
print(
    "부족한 전체 행 수:",
    (360 - incomplete_blocks["ROW_COUNT"]).sum()
)

incomplete_blocks

전체 기대 구간 수: 1472
불완전한 시간 구간 수: 129
부족한 전체 행 수: 46440


,STN,DATE,TIME_BLOCK,ROW_COUNT
0,400,20250703,06_11,0
1,400,20250712,00_05,0
2,400,20250712,06_11,0
3,400,20250712,12_17,0
4,400,20250715,06_11,0
...,...,...,...,...
124,101,20251218,12_17,0
125,101,20251219,00_05,0
126,101,20251225,06_11,0
127,101,20251229,12_17,0


### 6.3 누락 시간구간 재수집

시간구간별 수집 현황을 확인한 결과,
129개의 6시간 구간에서 매분자료가 전체 누락된 것을 확인하였습니다.

누락된 시간구간만 다시 API에 요청하여 데이터를 보완하며,
각 구간은 정상적인 360개 매분자료가 반환되는지 검증한 후 추가합니다.

In [31]:
recovery_list = []
recovery_failed = []

for i, row in incomplete_blocks.iterrows():
    stn = str(row["STN"])
    date = row["DATE"]
    time_block = row["TIME_BLOCK"]

    start_time, end_time = TIME_BLOCK_RANGES[time_block]

    tm1 = f"{date}{start_time}"
    tm2 = f"{date}{end_time}"

    print(
        f"[{i + 1}/{len(incomplete_blocks)}] "
        f"{stn} / {date} / {time_block}"
    )

    try:
        df = get_aws_minute_range(
            stn=stn,
            tm1=tm1,
            tm2=tm2
        )

        recovery_list.append(df)

    except (
        requests.exceptions.RequestException,
        RuntimeError
    ) as e:
        recovery_failed.append({
            "STN": stn,
            "DATE": date,
            "TIME_BLOCK": time_block,
            "ERROR": type(e).__name__
        })

    time.sleep(0.3)

[1/129] 400 / 20250703 / 06_11
[2/129] 400 / 20250712 / 00_05
[3/129] 400 / 20250712 / 06_11
[4/129] 400 / 20250712 / 12_17
[5/129] 400 / 20250715 / 06_11
[6/129] 400 / 20250718 / 18_23
[7/129] 400 / 20250722 / 18_23
[8/129] 400 / 20250723 / 12_17
[9/129] 400 / 20250726 / 06_11
[10/129] 400 / 20250731 / 00_05
[11/129] 400 / 20250801 / 18_23
[12/129] 400 / 20250802 / 00_05
[13/129] 400 / 20250802 / 18_23
[14/129] 400 / 20250803 / 12_17
[15/129] 400 / 20250805 / 18_23
[16/129] 400 / 20250806 / 00_05
[17/129] 400 / 20250806 / 06_11
[18/129] 400 / 20250806 / 12_17
[19/129] 400 / 20250806 / 18_23
[20/129] 400 / 20250807 / 00_05
[21/129] 400 / 20250813 / 06_11
[22/129] 400 / 20250824 / 12_17
[23/129] 400 / 20250825 / 00_05
[24/129] 400 / 20250825 / 06_11
[25/129] 400 / 20250828 / 06_11
[26/129] 400 / 20250901 / 18_23
[27/129] 400 / 20250902 / 00_05
[28/129] 400 / 20250907 / 00_05
[29/129] 400 / 20250908 / 18_23
[30/129] 400 / 20250913 / 12_17
[31/129] 400 / 20250918 / 00_05
[32/129] 400 / 20

In [32]:
if recovery_list:
    recovered_data = pd.concat(
        recovery_list,
        ignore_index=True
    )

    weather_minute_raw = pd.concat(
        [weather_minute_raw, recovered_data],
        ignore_index=True
    )

    # 혹시 모를 중복 제거
    weather_minute_raw = (
        weather_minute_raw
        .drop_duplicates(
            subset=["TM", "STN"],
            keep="last"
        )
        .reset_index(drop=True)
    )

print("재수집 성공 구간:", len(recovery_list))
print("재수집 실패 구간:", len(recovery_failed))
print("현재 전체 행 수:", len(weather_minute_raw))

재수집 성공 구간: 129
재수집 실패 구간: 0
현재 전체 행 수: 529920


### 6.4 최종 수집 결과 검증

누락 시간구간 재수집이 완료된 이후 전체 데이터를 다시 검증합니다.

전체 예상 행 수와 실제 행 수를 비교하고,
관측소·날짜·시간구간별 360개 매분자료가 모두 존재하는지 확인합니다.

또한 동일한 관측소와 시각 조합이 중복으로 존재하지 않는지 확인하여
최종 수집 데이터의 완전성을 검증합니다.

In [33]:
# 전체 예상 행 수
expected_total_rows = (
    len(weather_dates)
    * len(STATIONS)
    * 24
    * 60
)

print("예상 전체 행 수:", expected_total_rows)
print("실제 전체 행 수:", len(weather_minute_raw))


# 중복 확인
duplicate_count = weather_minute_raw.duplicated(
    subset=["STN", "TM"]
).sum()

print("중복 행 수:", duplicate_count)


# 6시간 구간별 행 수 재검증
minute_final_check = weather_minute_raw[["TM", "STN"]].copy()

minute_final_check["DATE"] = (
    minute_final_check["TM"].str[:8]
)

minute_final_check["HOUR"] = (
    minute_final_check["TM"]
    .str[8:10]
    .astype(int)
)

minute_final_check["TIME_BLOCK"] = (
    minute_final_check["HOUR"] // 6
).map({
    0: "00_05",
    1: "06_11",
    2: "12_17",
    3: "18_23"
})

final_block_count = (
    minute_final_check
    .groupby(["STN", "DATE", "TIME_BLOCK"])
    .size()
    .reindex(
        expected_index,
        fill_value=0
    )
    .reset_index(name="ROW_COUNT")
)

final_incomplete_blocks = final_block_count[
    final_block_count["ROW_COUNT"] != 360
]

print(
    "불완전한 시간 구간 수:",
    len(final_incomplete_blocks)
)

final_incomplete_blocks

예상 전체 행 수: 529920
실제 전체 행 수: 529920
중복 행 수: 0
불완전한 시간 구간 수: 0


,STN,DATE,TIME_BLOCK,ROW_COUNT


#### 확인 결과

2025년 7월부터 12월까지 서울 강남구와 강원 춘천시의
AWS 매분자료를 최종 검증한 결과 총 529,920개의 자료가 수집되었습니다.

관측소·날짜·6시간 구간별 데이터 개수를 확인한 결과
모든 구간에 360개의 매분자료가 존재하였으며,
관측소와 관측시각 기준 중복 데이터도 확인되지 않았습니다.

따라서 대상 기간과 관측소에 대한 AWS 매분자료 수집이 정상적으로 완료되었습니다.

## 7. 수집 데이터 저장

수집 및 검증이 완료된 기상 데이터를 원본 CSV 파일로 저장합니다.

일별 기상자료와 AWS 매분자료는 서로 다른 수집 단위를 가지므로
별도의 파일로 저장하며, Raw 데이터베이스 적재 전 원본 데이터로 활용합니다.

데이터 타입 변환, 결측값 처리 및 시간대 집계는 수행하지 않고
수집된 값을 현재 상태 그대로 보존합니다.

In [40]:
# 저장 파일 경로
daily_path = (
    WEATHER_RAW_DIR
    / "weather_daily_202507_202512.csv"
)

minute_path = (
    WEATHER_RAW_DIR
    / "weather_minute_202507_202512.csv"
)

# CSV 저장
weather_raw.to_csv(
    daily_path,
    index=False,
    encoding="utf-8-sig"
)

weather_minute_raw.to_csv(
    minute_path,
    index=False,
    encoding="utf-8-sig"
)

In [41]:
print("[저장 완료]")

print(
    f"일별 기상 데이터: "
    f"{daily_path.name} / "
    f"{len(weather_raw):,}행"
)

print(
    f"AWS 매분자료: "
    f"{minute_path.name} / "
    f"{len(weather_minute_raw):,}행"
)

print("\n저장 경로:")
print(WEATHER_RAW_DIR)

[저장 완료]
일별 기상 데이터: weather_daily_202507_202512.csv / 1,839행
AWS 매분자료: weather_minute_202507_202512.csv / 529,920행

저장 경로:
/Users/lee-jongyoon/Documents/bigcontest/00_data/weather


In [42]:
# 저장 파일 검증
daily_saved = pd.read_csv(
    daily_path,
    dtype=str
)

minute_saved = pd.read_csv(
    minute_path,
    dtype=str
)

print(
    "일별 기상 데이터:",
    daily_saved.shape
)

print(
    "AWS 매분자료:",
    minute_saved.shape
)

일별 기상 데이터: (1839, 9)
AWS 매분자료: (529920, 18)
